# Banking on Behaviour - Starter Notebook

This notebook shows you how to load the data and create a valid submission.

Feature engineering, model selection, and validation strategy are up to you.

**Target:** Predict `next_3m_txn_count` for each customer in Test.csv

**Metric:** RMSLE (Root Mean Squared Logarithmic Error)

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Core files
train = pd.read_csv('../data/raw/Train.csv')
test  = pd.read_csv('../data/raw/Test.csv')

print(f'Train: {train.shape}, Test: {test.shape}')
train.head()

Train: (8360, 2), Test: (3584, 1)


,UniqueID,next_3m_txn_count
0,00093e2d-9e1e-4061-ad27-a79b8ff9e165,129
1,0011d60f-a4e2-4333-81fc-2d557a82109b,16
2,0016f1e2-64c1-4c65-a668-1dc6bf3b5875,117
3,001aa3c5-632d-435e-a421-cc3615ccef4d,70
4,00298c6f-4f9d-4f28-b72c-ad0e56e9eb84,393


In [2]:
# Load transactions
txn = pd.read_parquet('../data/raw/transactions_features.parquet')
txn['TransactionDate'] = pd.to_datetime(txn['TransactionDate'])
max_date = txn['TransactionDate'].max()
print(f'Transactions: {len(txn):,} rows | Date range: {txn["TransactionDate"].min()} to {max_date}')
print(txn.columns.tolist())

Transactions: 18,017,073 rows | Date range: 2012-12-25 00:00:00 to 2015-10-31 00:00:00
['UniqueID', 'AccountID', 'TransactionDate', 'TransactionAmount', 'TransactionTypeDescription', 'TransactionBatchDescription', 'StatementBalance', 'IsDebitCredit', 'ReversalTypeDescription']


In [3]:
# ============================================================
# FEATURE BLOCK 1: Overall transaction stats (all time)
# ============================================================
feat_all = txn.groupby('UniqueID').agg(
    txn_count_all   = ('TransactionAmount', 'count'),
    txn_sum_all     = ('TransactionAmount', 'sum'),
    txn_mean_all    = ('TransactionAmount', 'mean'),
    txn_std_all     = ('TransactionAmount', 'std'),
    txn_median_all  = ('TransactionAmount', 'median'),
).reset_index()

# ============================================================
# FEATURE BLOCK 2: Debit vs Credit split
# ============================================================
# The competition notes: TransactionAmount is signed — negatives = debits, positives = credits
debits  = txn[txn['TransactionAmount'] < 0]
credits = txn[txn['TransactionAmount'] > 0]

feat_debit = debits.groupby('UniqueID').agg(
    debit_count = ('TransactionAmount', 'count'),
    debit_sum   = ('TransactionAmount', 'sum'),
    debit_mean  = ('TransactionAmount', 'mean'),
).reset_index()

feat_credit = credits.groupby('UniqueID').agg(
    credit_count = ('TransactionAmount', 'count'),
    credit_sum   = ('TransactionAmount', 'sum'),
    credit_mean  = ('TransactionAmount', 'mean'),
).reset_index()

print('Debit/credit features built')

Debit/credit features built


In [4]:
# ============================================================
# FEATURE BLOCK 3: Recency windows (6m, 3m, 1m)
# ============================================================
def window_features(df, days, suffix):
    cutoff = max_date - pd.Timedelta(days=days)
    sub = df[df['TransactionDate'] >= cutoff]
    feat = sub.groupby('UniqueID').agg(
        count = ('TransactionAmount', 'count'),
        sum   = ('TransactionAmount', 'sum'),
        mean  = ('TransactionAmount', 'mean'),
    ).reset_index()
    feat.columns = ['UniqueID', f'txn_count_{suffix}', f'txn_sum_{suffix}', f'txn_mean_{suffix}']
    return feat
feat_12m = window_features(txn, 365, '12m')
feat_6m = window_features(txn, 180, '6m')
feat_3m = window_features(txn, 90,  '3m')
feat_1m = window_features(txn, 30,  '1m')

print('Recency window features built')

Recency window features built


In [5]:
# ============================================================
# BLOCK 4: Month-by-month counts for last 6 months
# This gives the model the SHAPE of activity, not just totals
# ============================================================
txn['year_month'] = txn['TransactionDate'].dt.to_period('M')
monthly_counts = txn.groupby(['UniqueID', 'year_month']).size().reset_index(name='monthly_count')

# Pivot: each of last 6 months becomes a column
months_sorted = sorted(txn['year_month'].unique())[-6:]
month_labels  = {m: f'month_m{i+1}' for i, m in enumerate(months_sorted)}  # m1=oldest, m6=most recent

monthly_pivot = monthly_counts[monthly_counts['year_month'].isin(months_sorted)].copy()
monthly_pivot['month_label'] = monthly_pivot['year_month'].map(month_labels)
monthly_pivot = monthly_pivot.pivot(index='UniqueID', columns='month_label', values='monthly_count').reset_index().fillna(0)
monthly_pivot.columns.name = None

# Consistency stats across ALL months
all_monthly = txn.groupby(['UniqueID', 'year_month']).size().reset_index(name='mc')
consistency = all_monthly.groupby('UniqueID')['mc'].agg(
    monthly_mean   = 'mean',
    monthly_std    = 'std',
    monthly_max    = 'max',
    monthly_min    = 'min',
    monthly_median = 'median',
    active_months  = 'count',
).reset_index()
# Coefficient of variation — lower = more consistent customer
consistency['monthly_cv'] = consistency['monthly_std'] / (consistency['monthly_mean'] + 1)

print(f'Monthly pivot shape: {monthly_pivot.shape}')
print(f'Consistency shape: {consistency.shape}')

Monthly pivot shape: (11917, 7)
Consistency shape: (11944, 8)


In [6]:
# ============================================================
# BLOCK 5: Holiday season + trend + recency
# ============================================================
# Holiday: Nov/Dec/Jan (matches prediction window)
holiday_mask = txn['TransactionDate'].dt.month.isin([11, 12, 1])
feat_holiday = txn[holiday_mask].groupby('UniqueID').agg(
    holiday_txn_count = ('TransactionAmount', 'count'),
    holiday_txn_sum   = ('TransactionAmount', 'sum'),
    holiday_txn_mean  = ('TransactionAmount', 'mean'),
).reset_index()

# Trend: last 3m vs prior 3m
cutoff_3m = max_date - pd.Timedelta(days=90)
cutoff_6m = max_date - pd.Timedelta(days=180)
recent = txn[txn['TransactionDate'] >= cutoff_3m].groupby('UniqueID')['TransactionAmount'].count().rename('cnt_recent')
prior  = txn[(txn['TransactionDate'] >= cutoff_6m) & (txn['TransactionDate'] < cutoff_3m)].groupby('UniqueID')['TransactionAmount'].count().rename('cnt_prior')
trend_df = pd.concat([recent, prior], axis=1).reset_index().fillna(0)
trend_df['txn_trend_ratio'] = (trend_df['cnt_recent'] + 1) / (trend_df['cnt_prior'] + 1)
trend_df['txn_trend_diff']  = trend_df['cnt_recent'] - trend_df['cnt_prior']
trend_df = trend_df[['UniqueID', 'txn_trend_ratio', 'txn_trend_diff']]

# Also: 12m vs prior 12m trend
cutoff_12m = max_date - pd.Timedelta(days=365)
cutoff_24m = max_date - pd.Timedelta(days=730)
recent_12 = txn[txn['TransactionDate'] >= cutoff_12m].groupby('UniqueID')['TransactionAmount'].count().rename('cnt_12m')
prior_12  = txn[(txn['TransactionDate'] >= cutoff_24m) & (txn['TransactionDate'] < cutoff_12m)].groupby('UniqueID')['TransactionAmount'].count().rename('cnt_prev_12m')
trend_12m = pd.concat([recent_12, prior_12], axis=1).reset_index().fillna(0)
trend_12m['txn_trend_ratio_12m'] = (trend_12m['cnt_12m'] + 1) / (trend_12m['cnt_prev_12m'] + 1)
trend_12m = trend_12m[['UniqueID', 'txn_trend_ratio_12m']]

# Recency + tenure
last_txn  = txn.groupby('UniqueID')['TransactionDate'].max().reset_index()
last_txn['days_since_last_txn'] = (max_date - last_txn['TransactionDate']).dt.days
last_txn  = last_txn[['UniqueID', 'days_since_last_txn']]

first_txn = txn.groupby('UniqueID')['TransactionDate'].min().reset_index()
first_txn['customer_tenure_days'] = (max_date - first_txn['TransactionDate']).dt.days
first_txn = first_txn[['UniqueID', 'customer_tenure_days']]

# avg transactions per active month
avg_per_month = feat_all[['UniqueID', 'txn_count_all']].merge(consistency[['UniqueID', 'active_months']], on='UniqueID')
avg_per_month['avg_txn_per_month'] = avg_per_month['txn_count_all'] / avg_per_month['active_months']
avg_per_month = avg_per_month[['UniqueID', 'avg_txn_per_month']]

print('Block 5 done')

Block 5 done


In [7]:
# ============================================================
# BLOCK 6: Transaction TYPE breakdown (13 types)
# This is one of the most important unused signals
# ============================================================
if 'TransactionTypeDescription' in txn.columns:
    # Count per type per customer
    top_txn_types = txn['TransactionTypeDescription'].value_counts().nlargest(10).index
    txn_type_pivot = (txn[txn['TransactionTypeDescription'].isin(top_txn_types)]
        .groupby(['UniqueID', 'TransactionTypeDescription'])
        .size()
        .unstack(fill_value=0)
        .reset_index()
    )
    txn_type_pivot.columns = (['UniqueID'] +
        [f'txntype_{c.replace(" ", "_").replace("&","and")[:30]}'
         for c in txn_type_pivot.columns[1:]])

    # Share of each type (normalised)
    type_cols = [c for c in txn_type_pivot.columns if c != 'UniqueID']
    total = txn_type_pivot[type_cols].sum(axis=1).replace(0, 1)
    for c in type_cols:
        txn_type_pivot[f'{c}_pct'] = txn_type_pivot[c] / total

    print(f'TxnType features: {txn_type_pivot.shape}')
else:
    txn_type_pivot = None
    print('No TransactionTypeDescription column')

# Statement balance features (wealth proxy)
if 'StatementBalance' in txn.columns:
    bal_feats = txn.groupby('UniqueID')['StatementBalance'].agg(
        bal_mean   = 'mean',
        bal_median = 'median',
        bal_std    = 'std',
        bal_max    = 'max',
        bal_min    = 'min',
        bal_last   = 'last',
    ).reset_index()
    bal_feats['bal_cv'] = bal_feats['bal_std'] / (bal_feats['bal_mean'].abs() + 1)
    print(f'Balance features: {bal_feats.shape}')
else:
    bal_feats = None
    print('No StatementBalance column')


TxnType features: (11944, 21)
Balance features: (11944, 8)


In [ ]:
# ============================================================
# BLOCK 7: Financial features — time series exploitation
# The financials have 23 monthly snapshots — treat as time series
# ============================================================
fin = pd.read_parquet('../data/raw/financials_features.parquet')
fin['RunDate'] = pd.to_datetime(fin['RunDate'])
max_fin_date = fin['RunDate'].max()
fin['month_num'] = (fin['RunDate'].dt.year - 2013)*12 + fin['RunDate'].dt.month

# ── 1. All-time aggregates ───────────────────────────────────
fin_all = fin.groupby('UniqueID').agg(
    avg_income    = ('NetInterestIncome',  'mean'),
    total_income  = ('NetInterestIncome',  'sum'),
    std_income    = ('NetInterestIncome',  'std'),
    avg_revenue   = ('NetInterestRevenue', 'mean'),
    total_revenue = ('NetInterestRevenue', 'sum'),
    std_revenue   = ('NetInterestRevenue', 'std'),
).reset_index()

# ── 2. Recent snapshots (3m, 6m) ────────────────────────────
for months, label in [(3,'3m'), (6,'6m')]:
    cutoff = max_fin_date - pd.DateOffset(months=months)
    rf = fin[fin['RunDate'] >= cutoff].groupby('UniqueID').agg(
        avg_income  = ('NetInterestIncome',  'mean'),
        avg_revenue = ('NetInterestRevenue', 'mean'),
    ).reset_index()
    rf.columns = ['UniqueID', f'fin_avg_income_{label}', f'fin_avg_revenue_{label}']
    fin_all = fin_all.merge(rf, on='UniqueID', how='left')

# ── 3. MOST RECENT snapshot per product (Oct 2015 state) ────
last_snap = fin[fin['RunDate'] == max_fin_date].groupby(['UniqueID','Product']).agg(
    income_last  = ('NetInterestIncome',  'mean'),
    revenue_last = ('NetInterestRevenue', 'mean'),
).reset_index()
last_pivot = last_snap.pivot_table(
    index='UniqueID', columns='Product',
    values=['income_last','revenue_last'], aggfunc='mean'
).reset_index()
last_pivot.columns = ['UniqueID'] + [f'last_{v}_{p}' for v,p in last_pivot.columns[1:]]

# ── 4. Income SLOPE per product (trend direction) ────────────
from scipy import stats as sp_stats

def safe_slope(group):
    if len(group) < 3 or group['month_num'].nunique() < 2:
        return 0.0
    try:
        slope, _, r, _, _ = sp_stats.linregress(group['month_num'], group['NetInterestIncome'])
        return slope
    except:
        return 0.0

fin_recent = fin[fin['RunDate'] >= max_fin_date - pd.DateOffset(months=12)]
for prod in ['Transactional', 'Investments', 'Mortgages']:
    prod_df = fin_recent[fin_recent['Product'] == prod]
    slopes = prod_df.groupby('UniqueID').apply(safe_slope).reset_index()
    slopes.columns = ['UniqueID', f'slope_{prod[:5].lower()}_12m']
    last_pivot = last_pivot.merge(slopes, on='UniqueID', how='left')

# ── 5. Number of accounts and products ──────────────────────
n_accounts    = fin.groupby('UniqueID')['AccountID'].nunique().reset_index(name='n_accounts')
product_count = fin.groupby('UniqueID')['Product'].nunique().reset_index(name='num_products')

has_mortgage    = fin.groupby('UniqueID')['Product'].apply(
    lambda x: int('Mortgages' in x.values)).reset_index(name='has_mortgage')
has_investments = fin.groupby('UniqueID')['Product'].apply(
    lambda x: int('Investments' in x.values)).reset_index(name='has_investments')

# Per-product all-time income
top_products = fin['Product'].value_counts().nlargest(4).index
prod_income = fin[fin['Product'].isin(top_products)].groupby(
    ['UniqueID','Product'])['NetInterestIncome'].mean().unstack(fill_value=0).reset_index()
prod_income.columns = ['UniqueID'] + [f'prod_income_{c}' for c in prod_income.columns[1:]]

fin_features = (fin_all
    .merge(last_pivot,      on='UniqueID', how='left')
    .merge(prod_income,     on='UniqueID', how='left')
    .merge(product_count,   on='UniqueID', how='left')
    .merge(n_accounts,      on='UniqueID', how='left')
    .merge(has_mortgage,    on='UniqueID', how='left')
    .merge(has_investments, on='UniqueID', how='left')
)
print(f'Financial features: {fin_features.shape}')
print(fin_features.columns.tolist())


Financial features: (11377, 18)


In [9]:
# ============================================================
# BLOCK 8: Demographics — full feature set
# ============================================================
demo = pd.read_parquet('../data/raw/demographics_clean.parquet')
demo = demo.replace(['None', 'Not Disclosed / Unknown', 'Other / Unclassified'], np.nan)

demo['BirthDate'] = pd.to_datetime(demo['BirthDate'], errors='coerce')
ref_date = pd.to_datetime('2015-10-01')
demo['age'] = (ref_date - demo['BirthDate']).dt.days // 365
demo.loc[(demo['age'] < 18) | (demo['age'] > 100), 'age'] = np.nan
demo['age'] = demo['age'].fillna(demo['age'].median())

# AnnualGrossIncome — log transform, fill 0 income separately
demo['AnnualGrossIncome'] = demo['AnnualGrossIncome'].fillna(0)
demo['log_annual_income'] = np.log1p(demo['AnnualGrossIncome'])
demo['has_zero_income']   = (demo['AnnualGrossIncome'] == 0).astype(int)

# LowIncomeFlag as binary
demo['is_low_income'] = (demo['LowIncomeFlag'] == 'Y').astype(int)

# is_minor
demo['is_minor'] = (demo['ClientType'] == 'Individual – Minor').astype(int)
demo['is_foreign'] = (demo['ClientType'] == 'Foreign Individual').astype(int)

# Marital status
demo['is_married'] = (demo['MaritalStatus'] == 'Married').astype(int)

# Education (CertificationTypeDescription)
edu_order = {'Primary Education': 1, 'Secondary Education': 2,
             'Tertiary Education': 3, 'Postgraduate Education': 4}
demo['education_level'] = demo['CertificationTypeDescription'].map(edu_order).fillna(0)

# ContactPreference
demo['opted_in'] = (demo['ContactPreference'] == 'Opted In').astype(int)

# City — top 10 + Other
top_cities = demo['ResidentialCityName'].value_counts().nlargest(10).index
demo['ResidentialCityName'] = np.where(demo['ResidentialCityName'].isin(top_cities),
                                        demo['ResidentialCityName'], 'Other')

# Target encode categorical columns using TRAIN target
cat_cols = ['Gender', 'IncomeCategory', 'OccupationCategory', 'IndustryCategory',
            'ResidentialCityName', 'CustomerBankingType', 'MaritalStatus',
            'CustomerOnboardingChannel']

train_target = train[['UniqueID', 'next_3m_txn_count']].copy()
demo_with_target = demo.merge(train_target, on='UniqueID', how='left')
global_mean = train['next_3m_txn_count'].mean()
smoothing = 10

demo_selected = demo[['UniqueID', 'age', 'log_annual_income', 'has_zero_income',
                       'is_low_income', 'is_minor', 'is_foreign', 'is_married',
                       'education_level', 'opted_in'] + cat_cols].copy()

for col in cat_cols:
    stats = demo_with_target.groupby(col)['next_3m_txn_count'].agg(['mean', 'count'])
    stats['encoded'] = (stats['mean'] * stats['count'] + global_mean * smoothing) / (stats['count'] + smoothing)
    demo_selected[f'te_{col}'] = demo_selected[col].map(stats['encoded']).fillna(global_mean)

numeric_cols = ['UniqueID', 'age', 'log_annual_income', 'has_zero_income',
                'is_low_income', 'is_minor', 'is_foreign', 'is_married',
                'education_level', 'opted_in'] + [f'te_{c}' for c in cat_cols]
demo_encoded = demo_selected[numeric_cols].copy()
print(f'Demo features: {demo_encoded.shape}')


Demo features: (11944, 18)


In [10]:
# ============================================================
# NEW BLOCK A: Monthly AMOUNT stats
# Model knows count per month but not how big transactions are monthly
# ============================================================
monthly_amounts = txn.groupby(['UniqueID', 'year_month'])['TransactionAmount'].sum().reset_index()
monthly_amount_stats = monthly_amounts.groupby('UniqueID')['TransactionAmount'].agg(
    amt_monthly_mean = 'mean',
    amt_monthly_std  = 'std',
    amt_monthly_max  = 'max',
    amt_monthly_min  = 'min',
).reset_index()
monthly_amount_stats['amt_monthly_cv'] = (
    monthly_amount_stats['amt_monthly_std'] / (monthly_amount_stats['amt_monthly_mean'].abs() + 1)
)
print('Block A done:', monthly_amount_stats.shape)

Block A done: (11944, 6)


In [11]:
# ============================================================
# NEW BLOCK B: Timing patterns
# Weekday vs weekend, month-start vs month-end behaviour
# Salary earners transact heavily at month-start — strong predictor
# ============================================================
txn['day_of_week']      = txn['TransactionDate'].dt.dayofweek
txn['is_weekend']       = (txn['day_of_week'] >= 5).astype(int)
txn['is_month_end']     = (txn['TransactionDate'].dt.day >= 25).astype(int)
txn['is_month_start']   = (txn['TransactionDate'].dt.day <= 5).astype(int)

timing_feats = txn.groupby('UniqueID').agg(
    weekend_txn_pct     = ('is_weekend',        'mean'),
    month_end_txn_pct   = ('is_month_end',       'mean'),
    month_start_txn_pct = ('is_month_start',     'mean'),
).reset_index()
print('Block B done:', timing_feats.shape)

Block B done: (11944, 4)


In [12]:
# ============================================================
# NEW BLOCK C: Inactive months
# How many months did customer have ZERO transactions?
# Gaps signal churners; consistent customers have no gaps
# ============================================================
# Total months in dataset
total_months = txn['year_month'].nunique()

active_month_counts = txn.groupby('UniqueID')['year_month'].nunique().reset_index()
active_month_counts.columns = ['UniqueID', 'active_months_count']
active_month_counts['inactive_months']   = total_months - active_month_counts['active_months_count']
active_month_counts['pct_active_months'] = active_month_counts['active_months_count'] / total_months
print('Block C done:', active_month_counts.shape)

Block C done: (11944, 4)


In [13]:
# ============================================================
# NEW BLOCK D: Same-month-last-year activity
# Best proxy for holiday prediction window (Nov-Jan 2015)
# Use Nov-Jan 2014 (exactly 1 year prior)
# ============================================================
ly_mask = (
    txn['TransactionDate'].dt.month.isin([11, 12, 1]) &
    txn['TransactionDate'].dt.year.isin([2013, 2014])  # prior year Nov/Dec/Jan
)
feat_holiday_ly = txn[ly_mask].groupby('UniqueID').agg(
    holiday_ly_count = ('TransactionAmount', 'count'),
    holiday_ly_sum   = ('TransactionAmount', 'sum'),
    holiday_ly_mean  = ('TransactionAmount', 'mean'),
).reset_index()
print('Block D done:', feat_holiday_ly.shape)

Block D done: (11146, 4)


In [14]:
# ============================================================
# NEW BLOCK E: Large transaction flags
# Customers who make large transactions behave differently
# ============================================================
p75 = txn['TransactionAmount'].abs().quantile(0.75)
p95 = txn['TransactionAmount'].abs().quantile(0.95)

txn['is_large_txn']  = (txn['TransactionAmount'].abs() >= p75).astype(int)
txn['is_xlarge_txn'] = (txn['TransactionAmount'].abs() >= p95).astype(int)

large_txn_feats = txn.groupby('UniqueID').agg(
    large_txn_count  = ('is_large_txn',  'sum'),
    large_txn_pct    = ('is_large_txn',  'mean'),
    xlarge_txn_count = ('is_xlarge_txn', 'sum'),
    xlarge_txn_pct   = ('is_xlarge_txn', 'mean'),
).reset_index()
print('Block E done:', large_txn_feats.shape)

Block E done: (11944, 5)


In [15]:
def build_dataset(base_df):
    df = base_df.copy()

    for feat in [
        feat_all, feat_debit, feat_credit,
        feat_12m, feat_6m, feat_3m, feat_1m,
        monthly_pivot, consistency,
        feat_holiday, trend_df, trend_12m,
        last_txn, first_txn, avg_per_month,
        fin_features, demo_encoded,
        monthly_amount_stats, timing_feats,
        active_month_counts, feat_holiday_ly,
        large_txn_feats,
    ]:
        df = df.merge(feat, on='UniqueID', how='left')

    
    if txn_type_pivot is not None:
        df = df.merge(txn_type_pivot, on='UniqueID', how='left')
    if bal_feats is not None:
        df = df.merge(bal_feats, on='UniqueID', how='left')

    # Derived ratios
    df['debit_credit_ratio']    = (df['debit_count'].fillna(0) + 1) / (df['credit_count'].fillna(0) + 1)
    df['recent_activity_ratio'] = (df['txn_count_3m'].fillna(0) + 1) / (df['txn_count_6m'].fillna(0) + 1)
    df['activity_6m_12m_ratio'] = (df['txn_count_6m'].fillna(0) + 1) / (df['txn_count_12m'].fillna(0) + 1)
    df['holiday_share']         = df['holiday_txn_count'].fillna(0) / (df['txn_count_all'].fillna(0) + 1)
    df['holiday_ly_share']      = df['holiday_ly_count'].fillna(0) / (df['txn_count_all'].fillna(0) + 1)
    df['holiday_yoy_ratio']     = (df['holiday_ly_count'].fillna(0) + 1) / (df['holiday_txn_count'].fillna(0) + 1)

    df = df.fillna(0)
    return df

train_merged = build_dataset(train)
test_merged  = build_dataset(test)
print(f'Train: {train_merged.shape}')
print(f'Test:  {test_merged.shape}')


Train: (8360, 132)
Test:  (3584, 131)


In [ ]:
# ============================================================
# Model: Unified LightGBM + XGBoost blend with sample weights
# ============================================================
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
from xgboost import XGBRegressor
from sklearn.model_selection import KFold

FEATURE_COLS = [c for c in train_merged.columns if c not in ['UniqueID','next_3m_txn_count']]
X      = train_merged[FEATURE_COLS]
y      = train_merged['next_3m_txn_count']
y_log  = np.log1p(y)
X_test = test_merged[FEATURE_COLS]
print(f'Features: {len(FEATURE_COLS)} | Train rows: {len(X)}')

# Sample weights — upweight low-activity customers
avg_txn  = train_merged['avg_txn_per_month'].values
weights  = 1.0 / (np.log1p(avg_txn) + 1.0)
weights  = weights / weights.mean()
weights  = np.clip(weights, 0.3, 3.0)

lgbm_params = dict(
    n_estimators=3000, learning_rate=0.008, num_leaves=63,
    max_depth=-1, min_child_samples=20, subsample=0.75,
    subsample_freq=1, colsample_bytree=0.6,
    reg_alpha=0.05, reg_lambda=0.1,
    random_state=42, n_jobs=-1, verbose=-1,
)
xgb_params = dict(
    n_estimators=3000, learning_rate=0.008, max_depth=6,
    min_child_weight=20, subsample=0.75, colsample_bytree=0.6,
    reg_alpha=0.05, reg_lambda=0.1, early_stopping_rounds=100,
    random_state=42, n_jobs=-1, verbosity=0, tree_method='hist',
)

kf              = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds       = np.zeros(len(X))
test_preds_lgbm = np.zeros(len(X_test))
test_preds_xgb  = np.zeros(len(X_test))
rmsle_lgbm, rmsle_xgb = [], []

for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
    X_tr, X_val = X.iloc[tr_idx],     X.iloc[val_idx]
    y_tr, y_val = y_log.iloc[tr_idx], y_log.iloc[val_idx]
    w_tr        = weights[tr_idx]

    lgbm = LGBMRegressor(**lgbm_params)
    lgbm.fit(X_tr, y_tr, sample_weight=w_tr,
             eval_set=[(X_val, y_val)],
             callbacks=[early_stopping(150, verbose=False), log_evaluation(period=-1)])
    lv = lgbm.predict(X_val)
    test_preds_lgbm += lgbm.predict(X_test) / 5
    rl = np.sqrt(np.mean((lv - y_val.values)**2))
    rmsle_lgbm.append(rl)

    xgb = XGBRegressor(**xgb_params)
    xgb.fit(X_tr, y_tr, sample_weight=w_tr,
            eval_set=[(X_val, y_val)], verbose=False)
    xv = xgb.predict(X_val)
    test_preds_xgb += xgb.predict(X_test) / 5
    rx = np.sqrt(np.mean((xv - y_val.values)**2))
    rmsle_xgb.append(rx)

    blend = 0.6*lv + 0.4*xv
    oof_preds[val_idx] = blend
    rb = np.sqrt(np.mean((blend - y_val.values)**2))
    print(f'Fold {fold+1} | LGBM={rl:.4f} XGB={rx:.4f} Blend={rb:.4f} | iters={lgbm.best_iteration_}')

test_preds = 0.6*test_preds_lgbm + 0.4*test_preds_xgb
print(f'\nLGBM  CV: {np.mean(rmsle_lgbm):.4f} +/- {np.std(rmsle_lgbm):.4f}')
print(f'XGB   CV: {np.mean(rmsle_xgb):.4f} +/- {np.std(rmsle_xgb):.4f}')
print(f'Blend OOF: {np.sqrt(np.mean((oof_preds - y_log.values)**2)):.4f}')


Features: 130 | Train rows: 8360
Sample weights: min=0.55 max=2.36 mean=1.00
  Fold 1 LOW: n=552 RMSLE=0.5239
  Fold 1 MID: n=551 RMSLE=0.2907
  Fold 1 HIGH: n=569 RMSLE=0.3351
Fold 1 | LGBM=0.3956 XGB=0.3977 Blend=0.3958 | LGBM_iters=684

  Fold 2 LOW: n=552 RMSLE=0.4855
  Fold 2 MID: n=551 RMSLE=0.3594
  Fold 2 HIGH: n=569 RMSLE=0.2367
Fold 2 | LGBM=0.3739 XGB=0.3743 Blend=0.3734 | LGBM_iters=857

  Fold 3 LOW: n=553 RMSLE=0.5245
  Fold 3 MID: n=550 RMSLE=0.3291
  Fold 3 HIGH: n=569 RMSLE=0.2430
Fold 3 | LGBM=0.3833 XGB=0.3843 Blend=0.3830 | LGBM_iters=621

  Fold 4 LOW: n=552 RMSLE=0.5759
  Fold 4 MID: n=551 RMSLE=0.2785
  Fold 4 HIGH: n=569 RMSLE=0.2462
Fold 4 | LGBM=0.3953 XGB=0.3948 Blend=0.3946 | LGBM_iters=775

  Fold 5 LOW: n=552 RMSLE=0.4550
  Fold 5 MID: n=552 RMSLE=0.3049
  Fold 5 HIGH: n=568 RMSLE=0.2683
Fold 5 | LGBM=0.3523 XGB=0.3516 Blend=0.3514 | LGBM_iters=636

LGBM  CV: 0.3801 +/- 0.0161
XGB   CV: 0.3805 +/- 0.0167
Blend OOF: 0.3800


In [ ]:
# Feature importance
fi = pd.DataFrame({'feature': FEATURE_COLS, 'importance': lgbm.feature_importances_})
fi = fi.sort_values('importance', ascending=False)
print(fi.head(25).to_string(index=False))

              feature  importance
         txn_count_1m        1430
         txn_count_6m        1200
        txn_count_12m        1160
         txn_count_3m        1010
           monthly_cv         871
activity_6m_12m_ratio         742
  month_start_txn_pct         718
      txn_trend_ratio         713
    month_end_txn_pct         678
  txn_trend_ratio_12m         657
        holiday_share         649
             month_m3         614
             month_m5         574
          monthly_std         562
    holiday_yoy_ratio         562
             month_m6         526
         monthly_mean         500
                  age         489
 customer_tenure_days         487
   debit_credit_ratio         479
             month_m1         479
recent_activity_ratio         478
  days_since_last_txn         471
      weekend_txn_pct         463
     holiday_ly_share         458


## 4. Create a Valid Submission

Your submission must match the format of SampleSubmission.csv exactly.

In [17]:
# Zindi wants log1p(raw_counts) — NOT the raw counts
test_preds_log = test_preds  # test_preds is already in log1p space from the model!
# No need for expm1 at all

submission = pd.read_csv('../data/raw/SampleSubmission.csv')
submission['next_3m_txn_count'] = test_preds_log  # values like 3.2, 4.1, 5.6

submission.to_csv('submission_v5.csv', index=False)
print(submission['next_3m_txn_count'].describe())

count    3584.000000
mean        4.387367
std         1.170258
min         1.164132
25%         3.587884
50%         4.622461
75%         5.243941
max         6.852884
Name: next_3m_txn_count, dtype: float64


## Local Scoring

You can score your submission locally using the included evaluate.py script:



Note: PublicReference.csv is only available if you have it for local testing. On Zindi, scoring is automatic.